# The Black-Scholes-Merton Option Pricing Model

A from-scratch derivation and implementation of the Black-Scholes-Merton model, including
the PDE, closed-form formulas, Greeks, surface visualizations, and dynamic delta hedging.

## 1. Why This Formula Changed the World

In 1973, Fischer Black, Myron Scholes, and Robert Merton published what would become the most important equation in quantitative finance. Before their work, pricing options was largely guesswork. After it, options trading exploded --- the formula gave everyone a common language.

The key insight: **you do not need to know where the stock is going** to price an option. You only need to know **how much the stock wiggles** (its volatility). This is deeply counterintuitive --- you can price a bet on a stock's future without predicting that future.

### What You Will Learn

By the end of this notebook, you will understand:

1. How stock prices are modeled as a random walk (**Geometric Brownian Motion**).
2. Why randomness requires a corrected chain rule (**Ito's Lemma**).
3. How to eliminate all risk by continuously hedging (**the BSM PDE**).
4. The celebrated **BSM formula** --- what each piece means financially.
5. The **Greeks** --- how traders measure and manage risk.
6. **Delta hedging** in practice --- theory versus reality.

### The Analogy: Insurance Pricing Without Predicting Disasters

Imagine you run an insurance company. You do not need to predict exactly when hurricanes will hit to price flood insurance. You need to know:
- How volatile weather patterns are (analogous to $\sigma$)
- The time until the policy expires (analogous to $T$)
- What the policy covers (analogous to the strike $K$)

Similarly, BSM prices options using volatility, time, and the strike --- without predicting the stock's direction.

> **Key Concept:** The BSM formula gives the unique price at which no arbitrage is possible. If anyone charges a different price, a clever trader can construct a risk-free profit. The formula is not a prediction --- it is a consequence of the impossibility of free money.

## 2. How Stock Prices Move: Geometric Brownian Motion

### The Random Walk Story

Imagine you are watching a stock ticker. Each second, the price jiggles up or down by some small random amount. Over a day, thousands of these tiny movements accumulate into the price change you see.

This is the idea of a **random walk**, and the continuous-time version is called **Brownian motion**. For stock prices, we use a variant called **Geometric Brownian Motion (GBM)** that ensures prices stay positive (a stock cannot go below zero).

### The Mathematical Model

Under GBM, the stock price $S_t$ evolves according to:

$$dS_t = \mu S_t\, dt + \sigma S_t\, dW_t$$

Let us unpack each piece:

| Symbol | Name | Meaning | Typical Value |
|--------|------|---------|--------------|
| $\mu$ | Drift | Average return per year | 5--15% |
| $\sigma$ | Volatility | How much the stock wiggles per year | 15--40% |
| $dt$ | Time increment | An infinitesimal passage of time | --- |
| $dW_t$ | Brownian increment | A random shock, $\sim N(0, \sqrt{dt})$ | --- |
| $\mu S_t\, dt$ | Deterministic part | The "expected" drift | Small and predictable |
| $\sigma S_t\, dW_t$ | Random part | The "surprise" component | Dominates short-term |

### The Exact Solution

The SDE above has a closed-form solution:

$$S_T = S_0 \exp\!\left[\left(\mu - \frac{\sigma^2}{2}\right)T + \sigma W_T\right]$$

where $W_T \sim N(0, T)$ is a normal random variable.

> **Key Concept:** Under GBM, stock prices are **lognormally distributed**. The log of the stock price follows a normal distribution. This means:
> - Stock prices are always positive (you cannot exponentiate your way to a negative number).
> - The distribution has a right skew: stocks can go up a lot, but can only go down to zero.
> - The famous "bell curve" applies to log-returns, not to prices themselves.

### Why the $\sigma^2/2$ Correction?

Notice the drift in the exponent is $\mu - \sigma^2/2$, not $\mu$. This is called the **Ito correction** and comes from the fact that the exponential of a random variable does not equal the exponential of the mean. It is a mathematical consequence of Jensen's inequality: for a convex function like $\exp$, $E[e^X] > e^{E[X]}$.

In practical terms: if a stock has 20% expected return ($\mu$) and 30% volatility ($\sigma$), the median outcome is lower than the mean outcome. The $\sigma^2/2$ term accounts for this asymmetry.

> **Common Mistake:** Forgetting the $\sigma^2/2$ correction leads to prices that drift upward too fast on average. This is a frequent source of bugs in simulation code.

Let us simulate some GBM paths to build visual intuition.

In [ ]:
%matplotlib inline
import numpy as np
from scipy import stats, optimize
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
rng = np.random.default_rng(SEED)

ATOL = 1e-10
RTOL = 1e-6

PRIMARY   = 'steelblue'
SECONDARY = 'coral'
TERTIARY  = 'seagreen'
ACCENT    = 'gold'
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12, 'axes.grid': True, 'grid.alpha': 0.3})

In [ ]:
def simulate_gbm(S0, mu, sigma, T, n_steps, n_paths, rng):
    """Simulate paths of geometric Brownian motion.
    
    Uses the exact solution: S_t = S_0 * exp((mu - sigma^2/2)*t + sigma*W_t)
    rather than Euler discretization, so each path is exact.
    
    Returns
    -------
    t : array of shape (n_steps+1,)
    S : array of shape (n_paths, n_steps+1)
    """
    dt = T / n_steps
    t = np.linspace(0, T, n_steps + 1)
    
    # Increments of Brownian motion: each is N(0, sqrt(dt))
    dW = rng.normal(0, np.sqrt(dt), size=(n_paths, n_steps))
    
    # Cumulative sum gives W_t
    W = np.zeros((n_paths, n_steps + 1))
    W[:, 1:] = np.cumsum(dW, axis=1)
    
    # Exact GBM solution (not an approximation!)
    S = S0 * np.exp((mu - 0.5 * sigma**2) * t[np.newaxis, :] + sigma * W)
    
    return t, S


# Simulate and visualize
S0, mu, sigma, T = 100, 0.08, 0.2, 1.0
t, S = simulate_gbm(S0, mu, sigma, T, n_steps=252, n_paths=50, rng=rng)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i in range(50):
    axes[0].plot(t, S[i], alpha=0.3, linewidth=0.5, color=PRIMARY)
axes[0].plot(t, S0 * np.exp(mu * t), 'k--', linewidth=2, label=r'$E[S_t] = S_0 e^{\mu t}$')
axes[0].set_xlabel('Time (years)')
axes[0].set_ylabel('Stock Price')
axes[0].set_title('GBM Sample Paths')
axes[0].legend()

# Distribution of S_T
_, S_large = simulate_gbm(S0, mu, sigma, T, n_steps=1, n_paths=100000, rng=rng)
S_T = S_large[:, -1]
axes[1].hist(S_T, bins=100, density=True, alpha=0.6, color=PRIMARY, label='Simulated')

# Theoretical lognormal density
s_grid = np.linspace(S_T.min(), S_T.max(), 500)
log_mean = np.log(S0) + (mu - 0.5 * sigma**2) * T
log_std = sigma * np.sqrt(T)
pdf = stats.lognorm.pdf(s_grid, s=log_std, scale=np.exp(log_mean))
axes[1].plot(s_grid, pdf, color=SECONDARY, linewidth=2, label='Lognormal PDF')
axes[1].set_xlabel('$S_T$')
axes[1].set_ylabel('Density')
axes[1].set_title(f'Distribution of $S_T$ (T={T})')
axes[1].legend()

plt.tight_layout()
plt.show()

The left plot shows 50 possible paths of the stock price. Each path is one possible future. The dashed line shows the expected path, but individual paths can deviate enormously.

The right plot shows the **lognormal distribution** of the final stock price. Note the characteristic right skew: the stock can go up a lot (unbounded), but cannot go below zero. The theoretical density (orange) matches the simulation perfectly.

> **Key Concept:** Under GBM, stock prices are lognormally distributed. This means $\ln(S_T/S_0) \sim N\!\left((\mu - \sigma^2/2)T,\, \sigma^2 T\right)$. The distribution of returns is normal; the distribution of prices is lognormal. This distinction matters for everything that follows.

> **Important:** GBM is a simplification. Real stock prices exhibit fatter tails (more extreme moves), volatility clustering (calm periods and stormy periods), and jumps (sudden crashes). These violations motivate extensions like stochastic volatility models and jump-diffusion models. But GBM remains the starting point for all of options theory.

## 3. Ito's Lemma: The Chain Rule with a Twist

### Why Do We Need a New Chain Rule?

In regular calculus, if $f(x)$ is a function and $x$ changes by $dx$, then $f$ changes by $f'(x)\,dx$. Simple.

But in stochastic calculus, $x$ is a random process driven by Brownian motion, which has an unusual property: $(dW)^2 = dt$. This means that the second-order term in the Taylor expansion, which we would normally ignore, **survives**.

Think of it this way: in ordinary calculus, $(dx)^2$ is infinitesimally small compared to $dx$, so we throw it away. But Brownian motion is so "rough" that $(dW)^2$ is of the same order as $dt$ --- it cannot be ignored.

### The Formula

**Ito's Lemma**: If $X_t$ satisfies $dX_t = \mu_t\,dt + \sigma_t\,dW_t$, and $f(t, X)$ is a smooth function, then:

$$df = \left(\frac{\partial f}{\partial t} + \mu_t \frac{\partial f}{\partial X} + \frac{1}{2}\sigma_t^2 \frac{\partial^2 f}{\partial X^2}\right)dt + \sigma_t \frac{\partial f}{\partial X}\,dW_t$$

The extra $\frac{1}{2}\sigma_t^2 \frac{\partial^2 f}{\partial X^2}$ term is the **Ito correction** --- the "twist" that distinguishes stochastic calculus from ordinary calculus.

> **Key Concept:** Ito's Lemma is the chain rule of stochastic calculus. The extra second-derivative term appears because Brownian motion has "infinite variation" --- it wiggles so much that second-order effects matter. This correction term is precisely what creates the $\sigma^2/2$ adjustment in the GBM solution.

### Application: Deriving the BSM PDE

When we apply Ito's Lemma to an option price $V(t, S)$ where $S$ follows GBM, we get:

$$dV = \left(\frac{\partial V}{\partial t} + \mu S \frac{\partial V}{\partial S} + \frac{1}{2}\sigma^2 S^2 \frac{\partial^2 V}{\partial S^2}\right)dt + \sigma S \frac{\partial V}{\partial S}\,dW_t$$

The $dW_t$ term is the random component --- this is the risk in holding the option. The key insight of BSM is that this risk can be **perfectly hedged**.

## 4. The Black-Scholes PDE: Hedging Away All Risk

### The Key Idea: Delta Hedging

Here is the breakthrough insight. Suppose you hold an option $V(t, S)$ and want to eliminate all risk. You can do this by holding exactly $\Delta = \frac{\partial V}{\partial S}$ shares of stock in the opposite direction.

Consider the portfolio:

$$\Pi = V(t, S) - \Delta\, S$$

By Ito's Lemma, the change in this portfolio is:

$$d\Pi = \left(\frac{\partial V}{\partial t} + \frac{1}{2}\sigma^2 S^2 \frac{\partial^2 V}{\partial S^2}\right)dt$$

The $dW_t$ terms cancel! The portfolio is **riskless** over the infinitesimal time step.

### No Arbitrage Implies the PDE

Since the portfolio is riskless, it must earn the risk-free rate $r$:

$$d\Pi = r\Pi\,dt$$

Substituting and rearranging, we get the **Black-Scholes PDE**:

$$\boxed{\frac{\partial V}{\partial t} + rS\frac{\partial V}{\partial S} + \frac{1}{2}\sigma^2 S^2 \frac{\partial^2 V}{\partial S^2} = rV}$$

This is a partial differential equation that any European option price must satisfy, regardless of the specific payoff.

> **Key Concept:** Notice what is **not** in this equation: the stock's expected return $\mu$! The drift rate of the stock has vanished. This is the mathematical manifestation of the principle that option prices do not depend on where the stock is going --- only on how much it wiggles ($\sigma$).

### Boundary Conditions

The PDE alone is not enough --- we need boundary conditions to pin down a specific option:

- **Call**: $V(T, S) = \max(S - K, 0)$ at expiry
- **Put**: $V(T, S) = \max(K - S, 0)$ at expiry

Different boundary conditions give different option prices, but all satisfy the same PDE.

> **Common Mistake:** The BSM PDE is sometimes confused with the BSM formula. The PDE is the general equation that all option prices satisfy. The formula is the specific solution for European calls and puts. Exotic options satisfy the same PDE but with different boundary conditions and generally require numerical methods to solve.

## 5. The BSM Formulas: Solving the PDE

Solving the Black-Scholes PDE with the call payoff $\max(S_T - K, 0)$ gives the famous closed-form formulas.

### European Call Price

$$C = S_0\,N(d_1) - K e^{-rT}\,N(d_2)$$

### European Put Price

$$P = K e^{-rT}\,N(-d_2) - S_0\,N(-d_1)$$

where:

$$d_1 = \frac{\ln(S_0/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}, \quad d_2 = d_1 - \sigma\sqrt{T}$$

and $N(\cdot)$ is the standard normal CDF (the probability that a standard normal variable is less than $\cdot$).

### What Does Each Piece Mean?

The formula has a beautiful financial interpretation. Let us dissect the call price $C = S_0 N(d_1) - K e^{-rT} N(d_2)$:

| Piece | Financial Meaning |
|-------|-------------------|
| $S_0 N(d_1)$ | The **delta-weighted stock value**: the current stock price times the probability-weighted fraction you would receive |
| $K e^{-rT}$ | The **present value of the strike**: what you would pay at expiry, discounted to today |
| $N(d_2)$ | The (risk-neutral) **probability** that the option finishes in the money ($S_T > K$) |
| $N(d_1)$ | The **delta** of the option: how much stock you need to replicate it |
| $d_1$, $d_2$ | Standardized distances measuring how far "in the money" the option is, adjusted for volatility and time |

Think of the formula as: **"What I get" minus "What I pay"**, both weighted by the probability that the option finishes in the money and discounted to today.

> **Key Concept:** The quantity $N(d_2)$ is the risk-neutral probability that the call finishes in the money. If $N(d_2) = 0.65$, there is a 65% chance (in the risk-neutral world) that $S_T > K$. The quantity $N(d_1) = \Delta$ is the hedge ratio: to replicate a call, hold $N(d_1)$ shares of stock.

### Worked Example

Let $S_0 = 100$, $K = 100$, $r = 5\%$, $T = 1$ year, $\sigma = 20\%$.

**Step 1: Compute $d_1$ and $d_2$.**

$$d_1 = \frac{\ln(100/100) + (0.05 + 0.02)\times 1}{0.20 \times 1} = \frac{0 + 0.07}{0.20} = 0.35$$

$$d_2 = 0.35 - 0.20 = 0.15$$

**Step 2: Look up the normal CDF values.**

$$N(0.35) \approx 0.6368, \quad N(0.15) \approx 0.5596$$

**Step 3: Plug into the formula.**

$$C = 100 \times 0.6368 - 100 \times e^{-0.05} \times 0.5596 = 63.68 - 95.12 \times 0.5596 = 63.68 - 53.23 = \$10.45$$

> **Common Mistake:** Students sometimes use the stock's expected return $\mu$ instead of the risk-free rate $r$ in the formula. Remember: BSM prices in the risk-neutral world where all assets earn $r$.

Let us implement and verify this.

In [ ]:
def bsm_d1_d2(S, K, r, T, sigma):
    """Compute d1 and d2 for BSM formula."""
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return d1, d2


def bsm_call(S, K, r, T, sigma):
    """BSM European call price."""
    d1, d2 = bsm_d1_d2(S, K, r, T, sigma)
    return S * stats.norm.cdf(d1) - K * np.exp(-r * T) * stats.norm.cdf(d2)


def bsm_put(S, K, r, T, sigma):
    """BSM European put price."""
    d1, d2 = bsm_d1_d2(S, K, r, T, sigma)
    return K * np.exp(-r * T) * stats.norm.cdf(-d2) - S * stats.norm.cdf(-d1)


# Verify our hand calculation
S0, K, r, T, sigma = 100, 100, 0.05, 1.0, 0.2

C = bsm_call(S0, K, r, T, sigma)
P = bsm_put(S0, K, r, T, sigma)

print(f"BSM Call Price: {C:.6f}")
print(f"BSM Put Price:  {P:.6f}")
print(f"Put-Call Parity: C - P = {C - P:.6f}, S - K*exp(-rT) = {S0 - K*np.exp(-r*T):.6f}")

Our hand calculation gave $\$10.45$, and the code gives $\$10.4506$ --- close, with the difference from rounding the CDF lookups.

Let us also see how call and put prices vary with the stock price.

> **Key Concept:** The gap between the option price curve and the intrinsic value line is the **time value** --- the extra worth due to the possibility that the stock might move favorably before expiry. Time value is largest for at-the-money options and shrinks as options move deep in or out of the money.

In [ ]:
# Price as function of S
S_range = np.linspace(50, 150, 200)
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(S_range, bsm_call(S_range, K, r, T, sigma), color=PRIMARY, linewidth=2, label='Call')
ax.plot(S_range, bsm_put(S_range, K, r, T, sigma), color=SECONDARY, linewidth=2, label='Put')
ax.plot(S_range, np.maximum(S_range - K, 0), 'k--', alpha=0.3, label='Call intrinsic')
ax.plot(S_range, np.maximum(K - S_range, 0), 'k:', alpha=0.3, label='Put intrinsic')
ax.axvline(K, color='gray', alpha=0.3, linestyle='-')
ax.set_xlabel('Stock Price S')
ax.set_ylabel('Option Price')
ax.set_title('BSM Option Prices vs Spot')
ax.legend()
plt.tight_layout()
plt.show()

Notice how the option prices (solid lines) are always above their intrinsic values (dashed/dotted lines). The gap is the **time value**. For at-the-money options ($S \approx K = 100$), the time value is maximized --- there is maximum uncertainty about whether the option will finish in or out of the money.

For deep in-the-money options (call with $S \gg K$), the curve approaches the intrinsic value --- the option is almost certain to be exercised, so there is little "optionality" left.

## 6. The Greeks: Measuring Risk Sensitivities

Traders do not just care about an option's price --- they care about **how the price changes** when market conditions shift. These sensitivities are called the **Greeks** (because they are named after Greek letters).

### The Five Key Greeks

| Greek | Symbol | What It Answers | Formula (Call) |
|-------|--------|----------------|----------------|
| **Delta** | $\Delta$ | "If the stock moves \$1, how much does my option move?" | $N(d_1)$ |
| **Gamma** | $\Gamma$ | "How fast does my delta change?" | $\frac{n(d_1)}{S\sigma\sqrt{T}}$ |
| **Theta** | $\Theta$ | "How much do I lose per day to time decay?" | $-\frac{S\sigma n(d_1)}{2\sqrt{T}} - rKe^{-rT}N(d_2)$ |
| **Vega** | $\mathcal{V}$ | "If volatility rises 1%, how much does my option gain?" | $S\sqrt{T}\,n(d_1)$ |
| **Rho** | $\rho$ | "If interest rates rise 1%, how much does my option change?" | $KT e^{-rT} N(d_2)$ |

where $n(x) = \frac{1}{\sqrt{2\pi}}e^{-x^2/2}$ is the standard normal PDF.

### Financial Interpretations

**Delta** is the most important Greek. A delta of 0.6 means: "If the stock moves \$1, my call option moves about \$0.60." It is also (approximately) the probability the option finishes in the money.

**Gamma** tells you how *non-linear* the option is. High gamma means delta is changing fast --- you need to rebalance your hedge frequently. Gamma is highest for at-the-money options near expiry.

**Theta** is the "time decay" --- the cost of holding an option. Every day that passes, the option loses some value because there is less time for the stock to move favorably. Theta is the price you pay for the privilege of optionality.

**Vega** measures sensitivity to volatility. Since BSM assumes constant volatility but real volatility fluctuates, vega tells traders how exposed they are to volatility changes. A vega of 30 means a 1 percentage point increase in volatility (e.g., from 20% to 21%) increases the option price by about \$0.30.

**Rho** measures sensitivity to interest rates. It is usually the least important Greek for short-dated options.

> **Key Concept:** There is a fundamental trade-off between gamma and theta. Options with high gamma (which benefit from stock moves) also have high theta (which costs you daily). You cannot get the benefit of one without paying the cost of the other. This is the "gamma-theta trade-off" and it is central to options trading.

### Worked Example (Continuing from Above)

For our ATM call ($S = K = 100$, $\sigma = 20\%$, $T = 1$):
- $d_1 = 0.35$, so $\Delta = N(0.35) = 0.637$
- $n(d_1) = n(0.35) = 0.3752$
- $\Gamma = 0.3752 / (100 \times 0.20 \times 1) = 0.0188$
- $\text{Vega} = 100 \times 1 \times 0.3752 = 37.52$

In [ ]:
def bsm_greeks(S, K, r, T, sigma, option_type='call'):
    """Compute all BSM Greeks.
    
    Returns dict with keys: price, delta, gamma, theta, vega, rho.
    """
    d1, d2 = bsm_d1_d2(S, K, r, T, sigma)
    
    Nd1 = stats.norm.cdf(d1)
    Nd2 = stats.norm.cdf(d2)
    nd1 = stats.norm.pdf(d1)  # standard normal PDF
    
    if option_type == 'call':
        price = S * Nd1 - K * np.exp(-r * T) * Nd2
        delta = Nd1
        theta = (-S * sigma * nd1 / (2 * np.sqrt(T))
                 - r * K * np.exp(-r * T) * Nd2)
        rho = K * T * np.exp(-r * T) * Nd2
    else:
        Nmd1 = stats.norm.cdf(-d1)
        Nmd2 = stats.norm.cdf(-d2)
        price = K * np.exp(-r * T) * Nmd2 - S * Nmd1
        delta = Nd1 - 1         # put delta is negative
        theta = (-S * sigma * nd1 / (2 * np.sqrt(T))
                 + r * K * np.exp(-r * T) * Nmd2)
        rho = -K * T * np.exp(-r * T) * Nmd2
    
    # Gamma and Vega are the same for calls and puts
    gamma = nd1 / (S * sigma * np.sqrt(T))
    vega = S * np.sqrt(T) * nd1
    
    return {'price': price, 'delta': delta, 'gamma': gamma,
            'theta': theta, 'vega': vega, 'rho': rho}


# Display Greeks for our standard example
S0, K, r, T, sigma = 100, 100, 0.05, 1.0, 0.2

g_call = bsm_greeks(S0, K, r, T, sigma, 'call')
g_put  = bsm_greeks(S0, K, r, T, sigma, 'put')

print(f"BSM Greeks (S={S0}, K={K}, r={r}, T={T}, sigma={sigma})")
print(f"{'':10s} {'Call':>12s} {'Put':>12s}")
for name in ['price', 'delta', 'gamma', 'theta', 'vega', 'rho']:
    print(f"  {name:8s} {g_call[name]:12.6f} {g_put[name]:12.6f}")

### Reading the Numbers

For our at-the-money call ($S = K = 100$):

- **Delta = 0.64**: If the stock rises \$1, the call rises about \$0.64.
- **Gamma = 0.019**: Delta changes by 0.019 per \$1 stock move.
- **Theta = -6.41** (per year) or about **-\$0.025 per day**: the call loses 2.5 cents daily to time decay.
- **Vega = 37.52**: If volatility rises from 20% to 21%, the call gains about \$0.38.
- **Rho = 53.23**: If interest rates rise from 5% to 6%, the call gains about \$0.53.

Notice that **gamma and vega are the same for calls and puts** with the same parameters. This follows from put-call parity.

> **Important:** Theta is negative for long options (holders lose value over time) and positive for short options (sellers collect time decay). This is the "rent" that option sellers earn for bearing risk.

Let us visualize how each Greek varies with the stock price.

In [ ]:
# Greeks as function of S
S_range = np.linspace(60, 140, 300)
greek_names = ['delta', 'gamma', 'theta', 'vega', 'rho']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, name in enumerate(greek_names):
    call_vals = np.array([bsm_greeks(s, K, r, T, sigma, 'call')[name] for s in S_range])
    put_vals  = np.array([bsm_greeks(s, K, r, T, sigma, 'put')[name] for s in S_range])
    
    axes[i].plot(S_range, call_vals, color=PRIMARY, linewidth=2, label='Call')
    axes[i].plot(S_range, put_vals, color=SECONDARY, linewidth=2, label='Put')
    axes[i].axvline(K, color='gray', alpha=0.3)
    axes[i].set_xlabel('S')
    axes[i].set_title(name.capitalize())
    axes[i].legend(fontsize=9)

axes[5].set_visible(False)
plt.suptitle('BSM Greeks vs Spot Price', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

Key observations from the plots:

- **Delta**: Call delta goes from 0 to 1 as the stock goes from deep OTM to deep ITM (S-curve shape). Put delta is the mirror image, going from -1 to 0.
- **Gamma**: Peaked at ATM ($S \approx K$). This is where delta changes fastest.
- **Theta**: Most negative at ATM. Deep ITM/OTM options have less time value to lose.
- **Vega**: Also peaked at ATM. At-the-money options are most sensitive to volatility changes.
- **Rho**: Calls have positive rho (benefit from higher rates because the PV of the strike decreases). Puts have negative rho.

> **Key Concept:** All the "peaked at ATM" Greeks (gamma, theta, vega) reflect the same underlying idea: at-the-money options are the most "uncertain" --- it is a coin flip whether they finish in or out of the money. This uncertainty is where all the action is.

## 7. Greek Surfaces: How Greeks Change with Time

The Greeks depend on both the stock price and the time remaining until expiry. As expiry approaches, the Greeks behave more and more extremely for ATM options:

- **Delta** for ATM options snaps from 0.5 to either 0 or 1 right before expiry.
- **Gamma** becomes a spike near ATM at expiry --- a "gamma bomb."

> **Important:** Traders managing portfolios of options near expiry face "pin risk" --- if the stock is near the strike at expiry, gamma becomes enormous and the hedge becomes unstable. A small stock move can flip the option from worthless to valuable (or vice versa).

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

S_grid = np.linspace(70, 130, 80)
T_grid = np.linspace(0.05, 2.0, 80)
SS, TT = np.meshgrid(S_grid, T_grid)

# Compute delta and gamma surfaces for call
delta_surf = np.zeros_like(SS)
gamma_surf = np.zeros_like(SS)
for i in range(SS.shape[0]):
    for j in range(SS.shape[1]):
        g = bsm_greeks(SS[i,j], K, r, TT[i,j], sigma, 'call')
        delta_surf[i,j] = g['delta']
        gamma_surf[i,j] = g['gamma']

fig = plt.figure(figsize=(16, 6))

# Delta surface
ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_surface(SS, TT, delta_surf, cmap='coolwarm', alpha=0.8, edgecolor='none')
ax1.set_xlabel('S')
ax1.set_ylabel('T')
ax1.set_zlabel('Delta')
ax1.set_title('Call Delta Surface')

# Gamma surface
ax2 = fig.add_subplot(122, projection='3d')
ax2.plot_surface(SS, TT, gamma_surf, cmap='viridis', alpha=0.8, edgecolor='none')
ax2.set_xlabel('S')
ax2.set_ylabel('T')
ax2.set_zlabel('Gamma')
ax2.set_title('Call Gamma Surface')

plt.tight_layout()
plt.show()

The delta surface shows a smooth S-curve that becomes a sharp step function as $T \to 0$. The gamma surface shows a gentle hill that becomes a tall, narrow spike near expiry at $S = K$.

In [ ]:
# Heatmaps (easier to read than 3D)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im1 = axes[0].pcolormesh(SS, TT, delta_surf, cmap='coolwarm', shading='auto')
axes[0].set_xlabel('S')
axes[0].set_ylabel('T')
axes[0].set_title('Call Delta')
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].pcolormesh(SS, TT, gamma_surf, cmap='viridis', shading='auto')
axes[1].set_xlabel('S')
axes[1].set_ylabel('T')
axes[1].set_title('Call Gamma')
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()

The heatmap makes the "gamma ridge" clearly visible: a band of high gamma running along $S = K$ that intensifies as $T$ decreases toward zero. This is why near-expiry hedging is so challenging.

## 8. Dynamic Delta Hedging: Theory Meets Practice

### The Ideal vs. Reality

The BSM derivation assumes **continuous hedging** --- rebalancing the stock position at every instant. In practice, we can only rebalance at discrete times (daily, hourly, etc.).

Let us see what happens when we try to delta-hedge a short call position:

1. **Sell a call** at the BSM price (collect the premium).
2. **Buy $\Delta_t$ shares** at each rebalancing date to hedge.
3. At expiry, **close out**: deliver stock if the call is exercised, or do nothing.

If BSM were exactly right and we could hedge continuously, our P&L would be exactly zero --- the premium collected would exactly cover the hedging costs. With discrete hedging, there will be some residual P&L, whose variance decreases as we hedge more frequently.

> **Key Concept:** Delta hedging is like steering a car. If you could adjust the steering wheel continuously, you would follow the road perfectly. But if you can only adjust every few seconds, you will wobble. The more frequently you adjust, the smoother the ride. The "wobble" in hedging is the **hedging error**.

### What We Expect to See

- **Mean P&L near zero**: On average, BSM gets it right.
- **Variance decreasing with frequency**: More frequent hedging = less wobble.
- **Some skewness**: Large stock moves create asymmetric hedging errors.

In [ ]:
def delta_hedge_simulation(S0, K, r, T, sigma, n_rebalance, n_simulations, rng):
    """Simulate dynamic delta hedging of a short call.
    
    We sell a call, then hedge by holding delta shares at each rebalancing date.
    The P&L at expiry measures the hedging error.
    
    Returns
    -------
    pnl : array of shape (n_simulations,) - hedging P&L per simulation
    """
    dt = T / n_rebalance
    
    # Premium received from selling the call
    premium = bsm_call(S0, K, r, T, sigma)
    
    pnl = np.zeros(n_simulations)
    
    for sim in range(n_simulations):
        # Simulate one GBM path
        S = np.zeros(n_rebalance + 1)
        S[0] = S0
        Z = rng.normal(size=n_rebalance)
        for i in range(n_rebalance):
            S[i+1] = S[i] * np.exp((r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z[i])
        
        # Dynamic hedging
        cash = premium  # start with the premium from selling the call
        shares = 0.0
        
        for i in range(n_rebalance):
            tau = T - i * dt  # time remaining to expiry
            if tau > 1e-10:
                delta = stats.norm.cdf(bsm_d1_d2(S[i], K, r, tau, sigma)[0])
            else:
                delta = 1.0 if S[i] > K else 0.0
            
            # Rebalance: buy/sell shares to match delta
            trade = delta - shares
            cash -= trade * S[i]   # pay for shares bought (or receive for sold)
            shares = delta
            
            # Cash earns risk-free rate between rebalances
            cash *= np.exp(r * dt)
        
        # At expiry: liquidate stock position and pay off the call
        cash += shares * S[-1]
        payoff = max(S[-1] - K, 0)
        cash -= payoff
        
        pnl[sim] = cash
    
    return pnl


# Run simulations with different rebalancing frequencies
S0, K, r, T, sigma = 100, 100, 0.05, 1.0, 0.2
n_sims = 10000

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, n_reb in enumerate([12, 52, 252]):
    pnl = delta_hedge_simulation(S0, K, r, T, sigma, n_reb, n_sims, rng)
    
    axes[idx].hist(pnl, bins=80, density=True, alpha=0.7, color=PRIMARY)
    axes[idx].axvline(0, color='black', linewidth=1)
    axes[idx].axvline(pnl.mean(), color=SECONDARY, linestyle='--', linewidth=2,
                      label=f'Mean={pnl.mean():.4f}')
    axes[idx].set_xlabel('P&L')
    axes[idx].set_ylabel('Density')
    freq = {12: 'Monthly', 52: 'Weekly', 252: 'Daily'}[n_reb]
    axes[idx].set_title(f'{freq} Rebalancing (std={pnl.std():.4f})')
    axes[idx].legend(fontsize=9)

plt.suptitle('Delta Hedging P&L Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

The results confirm our expectations:

- **Mean P&L is near zero** for all frequencies: on average, the BSM price is "correct."
- **Standard deviation shrinks** as we rebalance more frequently: monthly (~\$2-3 spread) vs. daily (~\$0.50 spread).
- The distribution is roughly centered on zero but has some skew --- large stock moves create larger hedging errors.

In practice, daily rebalancing is standard. The remaining hedging error is the cost of not being able to trade continuously, and it is a key consideration in options market-making.

> **Important:** In the real world, transaction costs (bid-ask spreads, commissions) add another layer. More frequent hedging reduces hedging error but increases transaction costs. Optimal hedging frequency balances these two effects.

## 9. Limitations of BSM

The BSM model is the starting point, not the ending point. Its assumptions are violated in real markets, motivating a rich landscape of extensions.

| Assumption | Reality | Extension |
|-----------|--------|----------|
| Constant $\sigma$ | Implied vol varies by strike and maturity (smile/skew) | Local vol (Dupire), stochastic vol (Heston) |
| Continuous trading | Discrete hedging introduces error | Transaction cost models |
| No transaction costs | Bid-ask spreads and commissions | Leland's model |
| No jumps | Stocks can gap overnight or on news | Merton's jump-diffusion |
| Lognormal returns | Real returns have fat tails | Variance gamma, NIG models |
| Constant $r$ | Interest rates fluctuate | Hull-White, HJM models |

Despite these limitations, BSM remains the lingua franca of options markets. When a trader says "that option is trading at 25 vol," they mean the BSM implied volatility is 25%. The model's simplicity and analytical tractability make it indispensable as a benchmark.

> **Key Concept:** BSM is "wrong but useful" --- like Newtonian mechanics in a relativistic universe. For most practical purposes, it gives answers that are close enough, and its deviations from reality (the volatility smile) are themselves informative about market expectations.

## 10. References

1. Black, F., & Scholes, M. (1973). *The pricing of options and corporate liabilities*. Journal of Political Economy, 81(3), 637-654.
2. Merton, R. C. (1973). *Theory of rational option pricing*. Bell Journal of Economics and Management Science, 4(1), 141-183.
3. Hull, J. C. (2018). *Options, Futures, and Other Derivatives* (10th ed.). Pearson.
4. Shreve, S. E. (2004). *Stochastic Calculus for Finance II: Continuous-Time Models*. Springer.
5. Wilmott, P. (2006). *Paul Wilmott on Quantitative Finance* (2nd ed.). Wiley.
6. Taleb, N. N. (1997). *Dynamic Hedging: Managing Vanilla and Exotic Options*. Wiley.